# Gradient Descent in Torch

In [ ]:
!pip install librosa
!pip install scikit-image
!wget https://github.com/digitalmusicprocessing/Week12_GradientDescent/raw/refs/heads/main/doves.wav
!wget https://raw.githubusercontent.com/digitalmusicprocessing/Week12_GradientDescent/refs/heads/main/CS372.png

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import time
import skimage

## Exercise 1: Compute The Median

A median $m$ of a set $X$ is the number that minimizes the sum of absolute differences between $m$ and all elements in the set


## Exercise 2: Sound Images

In [ ]:
def get_mel_filterbank(win_length, sr, min_freq=50, max_freq=8000, n_bins=100):
    """
    Compute a mel-spaced filterbank and place it in a matrix
    
    Parameters
    ----------
    K: int
        Number of frequency bins
    sr: int
        The sample rate used to generate sdb
    min_freq: int
        The center of the minimum mel bin, in hz
    max_freq: int
        The center of the maximum mel bin, in hz
    n_bins: int
        The number of mel bins to use
    
    Returns
    -------
    ndarray(n_bins, n_win)
        The mel-spaced spectrogram
    """
    K = win_length//2+1
    print("K", K)
    # Space bin centers exponentially between min_freq and max_freq
    bins = np.logspace(np.log10(min_freq), np.log10(max_freq), n_bins+2)*win_length/sr
    bins = np.array(np.round(bins), dtype=int)
    Mel = np.zeros((n_bins, K), dtype=np.float32)
    for i in range(n_bins):
        i1 = bins[i]
        i2 = bins[i+1]
        if i1 == i2:
            i2 += 1
        i3 = bins[i+2]
        if i3 <= i2:
            i3 = i2+1
        tri = np.zeros(K)
        ## TODO: Create a triangle in the tri array which
        ## goes from 0 to 1 between index i1 and i2
        tri[i1:i2] = np.linspace(0, 1, i2-i1)
        ## and from 1 to 0 between index i2 and i3.
        tri[i2:i3] = np.linspace(1, 0, i3-i2)
        ## Then, place this triangle at row i of the Mel matrix
        Mel[i] = tri
    Mel = torch.from_numpy(Mel)
    return Mel


def get_stft(X, win_length):
    """
    Perform a Hann-windowed real STFT on batches of audio samples,
    assuming the hop length is half of the window length

    Parameters
    ----------
    X: torch.tensor(n_samples)
        Audio samples
    win_length: int
        Window length
    
    Returns
    -------
    S: torch.tensor(win_length//2+1, 1+2*(n_samples-win_length)//win_length))
        Real windowed STFT
    """
    n_samples = X.shape[0]
    hop_length = win_length//2
    T = (n_samples-win_length)//hop_length+1
    hann = torch.hann_window(win_length).to(X)
    hann = hann.view(1, win_length)

    ## Take out each overlapping window of the signal
    XW = torch.zeros(T, win_length).to(X)
    n_even = n_samples//win_length
    XW[0::2, 0:win_length] = X[0:n_even*win_length].view(n_even, win_length)
    n_odd = T - n_even
    XW[1::2, 0:win_length] = X[hop_length:hop_length+n_odd*win_length].view(n_odd, win_length)
    
    # Apply hann window and invert
    XW[:, 0:win_length] *= hann
    return torch.fft.rfft(XW, dim=-1).T

### Exercise 3: Beat It